# 多模态交互系统全流程开发与联调优化

---

## 课程导航

| 部分 | 内容 | 对应文件 |
|------|------|----------|
| **Part 1** | 本节定位 · 后端能力如何"送达"用户 | — |
| **Part 2** | 实时事件链路（本节核心） | `core/api/ws_api.py`、`core/infra/ws_manager.py` |
| **Part 3** | 前端三端与源码职责（合并为一章，纯架构） | `frontend/src/**` |
| **Part 4** | 接口层（REST API）系统讲解 · 鉴权 · 媒体 · 统计 | `core/api/*.py`、`core/infra/{auth,middleware}.py`、`core/service/analytics.py` |
| **Part 5** | 联调与稳定性 + 随堂练习（含答案） | 前后端联合 |

---

> **面向学员**：本课程的主线是**多模态大模型与智能体**。
> 因此本节**不是**"Vue 前端源码课"——前端只讲到"能看懂产品形态"的程度，
> 把篇幅还给真正的主线：**第 35–37 节产出的多模态结果与 Agent 结论，
> 是怎么实时送达用户与坐席的。**
>
> 上一节（第 37 节）让后端"会分工、会转人"；本节回答最后一公里：
> **这些能力怎么被用户看见？实时消息为什么不会丢？断线、多端登录怎么兜底？**

---
## Part 1：本节在架构中的位置 —— 后端能力如何"送达"用户
---

### 1.1 承上启下：能力有了，还要"送到人手上"

| 课 | 解决的问题 | 产出 |
|---|---|---|
| 第 35 节 | 能听会说、能看懂图 | 文本 / 结构化诊断 |
| 第 36 节 | 知识怎么进、怎么被找到 | 多模态 RAG 检索结果 |
| 第 37 节 | 会分工、会转人 | Agent 回复、转人工、工单 |
| **第 38 节** | **怎么送达用户与坐席** | **三端界面 + 实时事件链路** |

本节的主角**不是前端框架**，而是"**能力 → 事件 → 界面**"这条送达链路。

### 1.2 端到端实时数据流（本节的目录）

```
用户点「发送」（文字/语音/截图）
   │  WS: {type:"user_message", conversation_id, data:{...}}
   ▼
/ws/user 端点 → 会话归属校验 → 后台任务执行 session_flow（第 37 节编排）
   │                                    （后台执行，不阻塞消息循环）
   ▼  逐 token（Agent 流式产出）
{type:"bot_stream_delta", data:{delta}}          ← 第 37 节一线 Agent 的正文
   ▼  结束（含引用/附件/朗读语音）
{type:"bot_stream_end", data:{content, citations, attachments, audio_url}}
   │
   ▼
前端 handleWsMessage → 累加到占位气泡 → 定稿渲染
```

转人工时同理：`handoff_triggered`（用户端看到"正在转接"）→ 坐席端收到带摘要的
`handoff_triggered` → 坐席接入后 `agent_joined`（用户端）/ `queue_update`（坐席端）。

### 1.3 本节三条主线

| 主线 | 要回答的问题 | 关键代码 |
|------|--------------|----------|
| **前端轮廓** | 三端怎么分权？各端长什么样？ | `frontend/src/**`（只看目录与路由） |
| **实时事件链路** | 多模态/Agent 的产出怎么变成事件推给用户？ | `ws_api.py` + `ws_manager.py` |
| **稳定性** | 断线、多端登录、慢接口、消息丢失怎么兜底？ | 上述两者的异常分支 |

### 1.4 两条贯穿全节的统一约定

1. **事件契约统一**：前端只认 `type` 字段；前后端 `type` 集合必须严格对齐
   （Part 5 会用代码把两边"对一遍"，这是联调最有效的手段）。
2. **一条连接复用**：每个角色只建**一条** WebSocket，切换会话不重建连接、只发 `switch` 指令。

---
## 环境准备（本节通用）
---

> 本节所有代码单元都依赖下面这一次环境检查：确认 Python 与依赖就绪、
> 安装 Jupyter 内的 `asyncio` 兼容补丁，并定义 `read()` / `show()` 两个读源码小工具。

In [1]:
# ============================================================
# Cell 1：环境检查 + asyncio 兼容补丁
# ============================================================
import sys
import importlib
import re

print("=" * 60)
print("  环境检查")
print("=" * 60)
print(f"  Python 版本: {sys.version.split()[0]}")
print(f"  解释器路径: {sys.executable}")
print()

for pkg, imp in {
    "fastapi": "fastapi",    # 后端框架（WS 端点 / 路由）
    "uvicorn": "uvicorn",    # ASGI 服务器
    "pydantic": "pydantic",
}.items():
    try:
        importlib.import_module(imp)
        print(f"     [OK] {pkg}")
    except ImportError:
        print(f"     [NO] {pkg} 未安装")

from pathlib import Path
print()
print(f"  frontend/src 存在：        {Path('frontend/src').exists()}")
print(f"  frontend/dist 存在（构建产物）：{Path('frontend/dist').exists()}")


# ------------------------------------------------------------------
# 兼容补丁：让 asyncio.run() 在"已运行事件循环"的内核里也能用（幂等·可重入）
# ------------------------------------------------------------------
# 部分 Jupyter / IDE 内核会把单元格代码放在一个**正在运行的事件循环**里执行，
# 此时直接调用 asyncio.run() 会报：
#     RuntimeError: asyncio.run() cannot be called from a running event loop
# 做法：包一层 run_async()——没有运行中的循环时等价 asyncio.run；已有循环时，
# 把协程丢到"独立线程 + 新事件循环"执行再取回结果。
# 幂等设计：原始实现保存在 asyncio 模块属性上，并用标记位避免重复打补丁，
# 因此本单元被重复执行 / 被复制多份都不会互相干扰（不会自递归）。
import asyncio
import concurrent.futures


def _install_asyncio_run_compat() -> bool:
    """安装兼容补丁；已安装则返回 False（不重复打补丁）。"""
    if getattr(asyncio, "_mde_run_async_patched", False):
        return False

    asyncio._mde_orig_run = asyncio.run   # 原始实现挂在模块属性上，后续不会被覆盖

    def run_async(coro, **kwargs):
        """在 notebook 中安全执行协程（兼容"已运行事件循环"的内核）。"""
        orig = asyncio._mde_orig_run      # 每次取模块上的原始实现，确保不递归
        try:
            asyncio.get_running_loop()
        except RuntimeError:
            return orig(coro, **kwargs)   # 常规路径：无运行中的循环，直接跑
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: orig(coro, **kwargs)).result()

    asyncio.run = run_async
    asyncio._mde_run_async_patched = True
    return True


if _install_asyncio_run_compat():
    print("\n已启用 run_async 兼容：asyncio.run() 现在可用于运行中的事件循环内核。")
else:
    print("\nrun_async 兼容已启用（跳过重复补丁）。")


# ------------------------------------------------------------------
# 读源码小工具：全节通用（read / show）
# ------------------------------------------------------------------
# 后续所有"源码走读"单元都用这两个函数：read() 读全文，show() 打印指定行区间。
def read(path):
    return Path(path).read_text(encoding="utf-8")


def show(path, start=1, end=None, label=""):
    """打印文件 [start, end] 行（1-based，含端点）——用于展示关键源码片段。"""
    lines = read(path).splitlines()
    end = min(end or len(lines), len(lines))
    print("=" * 68)
    print(f"  {path}" + (f"  —— {label}" if label else "") + f"  （第 {start}~{end} 行 / 共 {len(lines)} 行）")
    print("=" * 68)
    for i in range(start, end + 1):
        print(f"{i:>4}| {lines[i-1]}")
    print()

  环境检查
  Python 版本: 3.13.7
  解释器路径: c:\Users\13691\Documents\Code\260601-GPKJ\07-MLM\.venv\Scripts\python.exe

     [OK] fastapi
     [OK] uvicorn
     [OK] pydantic

  frontend/src 存在：        True
  frontend/dist 存在（构建产物）：True

已启用 run_async 兼容：asyncio.run() 现在可用于运行中的事件循环内核。


---
## Part 2：实时事件链路—— 能力如何变成"事件"
---

### 2.1 为什么必须用 WebSocket

聊天是"**服务端主动找你**"的场景：Agent 的流式回复、转人工通知、坐席接入、队列变化……
HTTP 只能"一问一答"，要么轮询（浪费）、要么长连接。所以三个角色各建一条 WS 长连接。

### 2.2 事件总表：本项目的"通信契约"

前 37 节的产出，最终都落成下面这些 `type`。**这张表就是前后端的契约**：

| 事件 `type` | 方向 | 由谁发出（后端） | 含义 |
|---|---|---|---|
| `user_message` | 客户端→服务端 | — | 用户发消息（含文字/图片/语音 asset） |
| `bot_stream_delta` | 服务端→用户 | `session_flow` | **Agent 流式正文的一个增量** |
| `bot_stream_end` | 服务端→用户 | `session_flow` | 流式结束（含引用/图片附件/朗读语音） |
| `bot_reply` | 服务端→用户 | `session_flow` | 非流式短回复（寒暄等） |
| `generation_stopped` | 服务端→用户 | `ws_api` | 「停止生成」已取消该回合 |
| `handoff_triggered` | 服务端→用户/坐席 | `session_router` | **转人工**（坐席侧带交接摘要） |
| `agent_joined` | 服务端→用户 | `session_router` | 人工客服已接入 |
| `agent_message` | 双向 | `session_router` | 坐席的人工回复 |
| `return_to_bot` | 服务端→用户 | `session_router` | 已交还 AI |
| `queue_update` | 服务端→坐席 | `ws_api`/`session_router` | 待接入队列变化 |
| `agent_status` | 服务端→坐席 | `ws_api` | 坐席在线/忙碌/离线变化 |
| `stats_update` | 服务端→管理 | `analytics` | 数据看板实时快照 |
| `conversation_update` | 服务端→用户 | `session_flow` | 会话标题被更新 |
| `system` / `pong` | 服务端→客户端 | 多处 | 系统提示 / 心跳应答 |

### 2.3 把后端真正发出的事件"扫"出来

不用人工维护上面那张表——直接扫后端源码里所有 `{"type": "..."}`，得到权威清单。

In [2]:
# ============================================================
# Cell 2：从后端源码提取所有事件 type
# ============================================================
BACKEND_FILES = [
    "core/infra/ws_manager.py",
    "core/api/ws_api.py",
    "core/service/session_flow.py",
    "core/service/session_router.py",
    "core/service/analytics.py",
]

type_pat = re.compile(r'"type":\s*"([a-z_]+)"')
backend_types = set()
for p in BACKEND_FILES:
    backend_types |= set(type_pat.findall(read(p)))

print("=" * 68)
print(f"  后端发出的事件 type（共 {len(backend_types)} 个）")
print("=" * 68)
for t in sorted(backend_types):
    print(f"    - {t}")
print()
print("  这些 type 就是前后端的'通信契约'：前端只认它，不认识就什么都不做。")

  后端发出的事件 type（共 16 个）
    - agent_joined
    - agent_message
    - agent_status
    - bot_reply
    - bot_stream_delta
    - bot_stream_end
    - conversation_update
    - generation_stopped
    - handoff_triggered
    - mode_changed
    - pong
    - queue_update
    - return_to_bot
    - stats_update
    - system
    - user_message

  这些 type 就是前后端的'通信契约'：前端只认它，不认识就什么都不做。


#### 事件从哪来、推给谁（生产 → 推送全景）

事件不是凭空出现，每条都由某个后端模块产生、再经 `ws_manager` 广播给特定角色：

| 事件 | 产生方 | 推给谁 |
|---|---|---|
| `bot_stream_delta` / `bot_stream_end` | `ws_api`（消费 LLM 流） | 该用户 |
| `handoff_triggered` | `session_router.trigger_handoff` | 用户 + 全体坐席 |
| `agent_joined` / `return_to_bot` | `session_router`（坐席操作） | 该用户 |
| `queue_update` | `session_router`（接入 / 转回） | 全体坐席 |
| `conversation_update` | `session_router`（新建 / 关闭） | 该用户 |
| `stats_update` | `analytics.push_stats_to_admins` | 全体管理员 |

`session_router` 是**事件的主要产地**，`ws_manager` 是唯一的**投递口**——所有推送都收敛到
`send_to_user` / `send_to_agents` / `broadcast_to_admins` 三个方法，前端因此只需认 `type`。


### 2.4 服务端：三个 WS 端点

| 端点 | 角色 | 客户端→服务端 | 服务端→客户端 |
|---|---|---|---|
| `/ws/user` | `user` | `user_message` / `stop` / `switch` / `ping` | `bot_stream_delta` / `bot_stream_end` / `bot_reply` / `handoff_triggered` / `agent_joined` / `return_to_bot` / `conversation_update` / `system` / `pong` |
| `/ws/agent` | `agent` | `agent_message` / `join` / `status` / `ping` | `queue_update` / `handoff_triggered` / `user_message` / `agent_message` / `agent_status` / `pong` |
| `/ws/admin` | `admin` | `refresh` / `ping` | `stats_update` / `pong` |

**握手鉴权**：浏览器的 WebSocket **无法自定义请求头**，所以 token 只能放 URL 查询参数
（`/ws/user?token=xxx`）。端点进入时先校验：未登录 `close(4401)`、角色不符 `close(4403)`——
此时还没 `accept()`，直接 close 就是"拒绝握手"。4400 段是**应用自定义关闭码**。

#### 关闭码约定：4400 段是应用自定义

| 关闭码 | 含义 | 触发点 |
|---|---|---|
| `4401` | 未登录 | 握手时 token 缺失 / 失效（`_authenticate`） |
| `4403` | 角色不符 | token 有效但角色与端点不匹配 |
| `4409` | 账号在别处登录 | `register_*` 发现同账号已有连接，踢掉旧连接 |

它们与 HTTP 的 `401 / 403` 一一对应，只是 WS 走握手期 `close()`、HTTP 走响应状态码。

#### 分组广播：三个投递口

| 方法 | 收件人 | 典型事件 |
|---|---|---|
| `send_to_user(user_id, msg)` | 单个用户连接 | `bot_reply`、`agent_joined` |
| `send_to_agents(msg)` | 所有在线坐席 | `queue_update`、`handoff_triggered` |
| `broadcast_to_admins(msg)` | 所有管理员 | `stats_update` |

> 目标离线时这三个方法**静默返回 False、不抛异常**——一条推送失败不能拖垮整轮广播。


In [3]:
# ============================================================
# Cell 3：ws_api 端点清单 + 握手鉴权源码
# ============================================================
import ast

tree = ast.parse(read("core/api/ws_api.py"))
print("=" * 68)
print("  core/api/ws_api.py 路由端点")
print("=" * 68)
for node in tree.body:
    if isinstance(node, ast.AsyncFunctionDef):
        for dec in node.decorator_list:
            if isinstance(dec, ast.Call) and getattr(dec.func, "attr", "") == "websocket":
                route = dec.args[0].value if dec.args else "?"
                print(f"  第 {node.lineno:>3} 行  @websocket({route!r})  ->  {node.name}()")
        
print()
show("core/api/ws_api.py", 45, 62, "_authenticate —— token + 角色校验")
show("core/api/ws_api.py", 106, 139, "ws_user 消息循环要点：后台任务 + 停止生成")

  core/api/ws_api.py 路由端点
  第  86 行  @websocket('/ws/user')  ->  ws_user()
  第 166 行  @websocket('/ws/agent')  ->  ws_agent()
  第 247 行  @websocket('/ws/admin')  ->  ws_admin()

  core/api/ws_api.py  —— _authenticate —— token + 角色校验  （第 45~62 行 / 共 285 行）
  45| async def _authenticate(ws: WebSocket, roles: tuple[str, ...]) -> Optional[dict]:
  46|     """握手鉴权：校验 token 与角色，通过返回用户字典，否则关闭连接并返回 None。
  47| 
  48|     调用方：三个 ws 端点进入时的第一件事。
  49|     - token 从 URL 查询参数取（WebSocket 无法像 HTTP 那样自定义请求头）；
  50|     - 未登录：close(code=4401)；
  51|     - 角色不在允许列表：close(code=4403)。
  52|     注意：此时尚未 accept()，直接 close 即为拒绝握手；调用方据返回 None 提前 return。
  53|     """
  54|     token = ws.query_params.get("token", "")
  55|     user = await get_current_user(token)
  56|     if not user:
  57|         await ws.close(code=4401)
  58|         return None
  59|     if user["role"] not in roles:
  60|         await ws.close(code=4403)
  61|         return None
  62|     return user

  core/api/ws_api.py  —— ws_user

### 2.5 服务端：连接管理与"消息不丢"的三个关键点

`WSManager` 用三张结构分别保存连接（用户 / 坐席按 id，管理员用列表）。
真正值钱的是**两个真实 Bug 的修复**：

**Bug 1：同账号多端互踢死循环。**
旧版被踢端发现断开后自动重连，又把新端顶掉，来回踢。
修复：踢旧连接时带 **`4409` 关闭码 + 原因**，前端据此停止重连。

**Bug 2：旧连接的 `finally` 误删新连接。**
重连时序是：新连接注册 → 旧连接被关闭 → 旧连接 `finally` 调 `unregister_user(user_id)`。
若按 `user_id` **无条件 pop**，就把刚注册的新连接删了——此后所有推送（`bot_reply` /
`agent_message`）被静默丢弃，而客户端连接看起来正常、也不会重连（最难查的一类故障）。

修复：注销时**校验身份**——只有"映射里还是我这条连接"才移除：

```python
def unregister_user(self, user_id, ws):
    if self.user_connections.get(user_id) is ws:   # 关键：比我是不是仍是当前连接
        self.user_connections.pop(user_id, None)
```

> 同样的修复也用在 `GenerationRegistry.discard`：上一回合的 done 回调可能在新回合登记之后
> 才执行，若按 `conv_id` 无条件 pop，会把新回合的 task 误删，导致新回合「停止生成」失效。

#### 心跳：让它自己发现「尸体连接」

TCP 层常常不能及时感知半开连接（网络闪断、拔网线）。本项目前端定时发 `ping`、服务端回 `pong`；
一旦 `send_json` 抛异常，就在广播时把该连接**剔除**，避免向「尸体连接」反复推送。
**心跳的价值不在保活，而在于让死连接尽快暴露。**


In [4]:
# ============================================================
# Cell 4：WSManager 注册 / 注销源码
# ============================================================
import core.infra.ws_manager as wsm

show("core/infra/ws_manager.py", 20, 62, "连接表 + register/unregister（含身份校验）")

  core/infra/ws_manager.py  —— 连接表 + register/unregister（含身份校验）  （第 20~62 行 / 共 135 行）
  20| class WSManager:
  21|     def __init__(self) -> None:
  22|         self.user_connections: dict[str, WebSocket] = {}   # user_id -> ws
  23|         self.agent_connections: dict[str, WebSocket] = {}  # agent_id -> ws
  24|         self.admin_connections: list[WebSocket] = []
  25| 
  26|     # ------------------------------------------------------------------
  27|     # 注册 / 注销
  28|     # ------------------------------------------------------------------
  29|     async def register_user(self, user_id: str, ws: WebSocket) -> None:
  30|         old = self.user_connections.get(user_id)
  31|         if old is not None and old is not ws:
  32|             await self._safe_close(old, code=WS_CLOSE_KICKED, reason="账号已在别处登录")
  33|         self.user_connections[user_id] = ws
  34| 
  35|     def unregister_user(self, user_id: str, ws: WebSocket) -> None:
  36|         """仅当映射里仍是这条连接时才移除。
  37| 
 

In [5]:
# ============================================================
# Cell 5：实测 —— 互踢 + 身份校验注销（离线，真跑 ws_manager）
# ============================================================
class FakeWS:
    """最小可用的 WebSocket 替身：只实现被调用到的两个方法。"""

    def __init__(self, name):
        self.name = name
        self.closed = None
        self.sent = []

    async def send_json(self, message):
        self.sent.append(message)

    async def close(self, code=1000, reason=""):
        self.closed = (code, reason)


async def demo_ws_manager():
    m = wsm.WSManager()
    old = FakeWS("旧连接")
    await m.register_user("u1", old)

    new = FakeWS("新连接")
    await m.register_user("u1", new)              # 同账号再次登录
    print(f"  ① 旧连接收到关闭：{old.closed}   （应为 {wsm.WS_CLOSE_KICKED}）")
    print(f"     当前映射已指向新连接：{m.user_connections['u1'] is new}")

    m.unregister_user("u1", old)                  # 旧连接的 finally 触发
    print(f"  ② 旧连接 finally 之后，新连接仍在：{m.user_connections.get('u1') is new}")

    ok = await m.send_to_user("u1", {"type": "bot_reply"})
    print(f"  ③ 推送命中新连接：{ok}，新连接收到：{new.sent}")


asyncio.run(demo_ws_manager())
print()
print("  结论：先踢旧、再注册新、注销校验身份 —— 三者缺一，就会出现'连接在但收不到消息'。")

  ① 旧连接收到关闭：(4409, '账号已在别处登录')   （应为 4409）
     当前映射已指向新连接：True
  ② 旧连接 finally 之后，新连接仍在：True
  ③ 推送命中新连接：True，新连接收到：[{'type': 'bot_reply'}]

  结论：先踢旧、再注册新、注销校验身份 —— 三者缺一，就会出现'连接在但收不到消息'。


### 2.6 服务端：Agent 流式回合为什么要放"后台任务" + 如何"停止生成"

`/ws/user` 里，AI 回合**不能阻塞消息循环**——否则生成期间用户再也发不出消息。
所以用 `asyncio.create_task` 丢后台，并把句柄登记到 `GenerationRegistry`，
这样用户点「停止」时能按 `conversation_id` **精准取消**该回合（中断流式输出）。

#### 三个容易踩的细节

1. **必须保存 task 引用**：`asyncio.create_task()` 的返回值若无人引用，可能被 GC 回收、任务中途消失；
   所以登记进 `GenerationRegistry`。
2. **异常也要收尾**：回合抛异常时同样要发 `bot_stream_end`（带错误说明）并清理登记，
   否则前端会永远停在「思考中」。
3. **停止生成是三态**：命中在跑的回合 → 取消成功；回合已结束 → 无事发生；
   未登记的会话 → 返回 `False`。前端据此决定是否复位输入区。


In [6]:
# ============================================================
# Cell 6：实测 GenerationRegistry（停止生成 + 身份校验丢弃）
# ============================================================
class FakeTask:
    def __init__(self):
        self.cancelled = False
        self._done = False

    def done(self):
        return self._done

    def cancel(self):
        self.cancelled = True
        self._done = True


async def demo_registry():
    g = wsm.GenerationRegistry()
    running = FakeTask()
    g.register("conv-A", running)
    print(f"  ① 取消 conv-A 的回合：{g.cancel('conv-A')}，任务被取消：{running.cancelled}")

    # 模拟：上一回合的 done 回调在新回合登记之后才执行
    old_round, new_round = FakeTask(), FakeTask()
    g.register("conv-B", new_round)
    g.discard("conv-B", old_round)                # 旧回合的 task，不是当前登记的那一个
    print(f"  ② 旧回合 discard 后，新回合仍在登记：{'conv-B' in g._tasks}")

    g.discard("conv-B", new_round)
    print(f"  ③ 新回合 discard 后已移除：{'conv-B' not in g._tasks}")


asyncio.run(demo_registry())
print()
print("  结论：注册表按 id 存，但'移除'必须校验是不是自己那条 —— 否则会误删新回合的'停止'能力。")

  ① 取消 conv-A 的回合：True，任务被取消：True
  ② 旧回合 discard 后，新回合仍在登记：True
  ③ 新回合 discard 后已移除：True

  结论：注册表按 id 存，但'移除'必须校验是不是自己那条 —— 否则会误删新回合的'停止'能力。


---
## Part 3：前端三端与源码职责
---

> 这一章**不写代码**：只回答两个问题——前端被分成了哪几层（架构），
> 以及每个源文件到底实现了什么（类 / 函数级职责）。

### 3.1 三端与角色

一个后端，服务三类人，前端对应三个入口：

| 端 | 路由 | 角色 | 干什么 |
|---|---|---|---|
| 用户端（员工） | `/chat` | `user` | 文字 / 语音 / 截图提问，看 AI 带图回复与参考来源 |
| 坐席端（人工客服） | `/agent` | `agent` | 待接入队列、接入对话、看 AI 摘要与辅助推荐 |
| 管理后台 | `/admin/*` | `admin` | 数据看板、知识库、工单、日志、评测、用户 |

### 3.2 前端工程分层

`frontend/src/` 按「入口 → 路由 → 状态 → 网络 → 布局 → 视图 → 组件」分层，
每层只干一件事，依赖单向向下：

```
frontend/src/
├── main.js / App.vue      入口：挂载 Vue / Pinia / Router / Element Plus
├── router/index.js        路由表 + meta.roles 权限守卫
├── store/user.js          Pinia：token / 用户信息（localStorage 持久化）
├── api/http.js            axios：注入 token、统一处理 401、媒体地址补 token
├── api/ws.js              WebSocket 封装：重连 / 缓冲 / 互踢
├── layouts/               UserLayout / AgentLayout / AdminLayout
├── components/            6 个可复用组件
├── views/                 Chat.vue / AgentConsole.vue / admin/*
└── utils/format.js        时间 / 文本格式化
```

### 3.3 目录明细（真实文件清单）

| 层 | 文件 |
|---|---|
| 入口 | `main.js`、`App.vue` |
| 路由 | `router/index.js` |
| 状态 | `store/user.js` |
| 网络 | `api/http.js`、`api/ws.js` |
| 布局 | `layouts/UserLayout.vue`、`AgentLayout.vue`、`AdminLayout.vue` |
| 组件 | `components/{MessageBubble,ImageAttachment,CitationList,ConversationList,ModeBanner,BaseChart}.vue` |
| 视图 | `views/{Login,Chat,AgentConsole}.vue`、`views/admin/{Dashboard,Knowledge,Tickets,Logs,ToolLogs,Users,Eval}.vue` |
| 工具 | `utils/format.js` |

### 3.4 权限守卫：一张路由表 + `meta.roles`

三端共用**同一个路由表**，靠 `meta.roles` 区分权限：

| 路径 | 角色 | 说明 |
|---|---|---|
| `/login` | 公开 | `meta.public`；已登录再访问则按角色导流 |
| `/chat` | `user` | `UserLayout` 包裹 |
| `/agent` | `agent` | `AgentLayout` 包裹 |
| `/admin/*` | `admin` | `AdminLayout` + 7 个子页 |

`router.beforeEach` 把 `to.matched` 上所有记录的 `roles` **合并**校验：父路由（Layout）
声明角色、子路由继承，**新增页面不用重复写角色**。菜单渲染与守卫用同一份 `roles`，
从根上避免「菜单能点、进去 403」。

### 3.5 实时接收：前端只认 `type`

用户端与坐席端各有一个 `handleWsMessage(msg)`，按 `msg.type` 分发：

| 端 | 关注事件 |
|---|---|
| 用户端 `Chat.vue` | `bot_stream_delta` / `bot_stream_end` / `bot_reply` / `handoff_triggered` / `agent_joined` / `return_to_bot` / `conversation_update` |
| 坐席端 `AgentConsole.vue` | `queue_update` / `handoff_triggered` / `user_message` / `agent_message` / `agent_status` |
| 管理端 `Dashboard.vue` | `stats_update` |

**这就是「事件契约」在前端的落点**——不认识的消息类型直接忽略。前端处理的事件集合
必须是后端发出集合的**子集**，不一致就说明有 Bug（Part 5 会用代码做差集检查）。

### 3.6 前端实时链路的四个要点（一句话各一条）

| 要点 | 做什么 | 为什么 |
|---|---|---|
| **一条连接复用** | 每角色只建一条 WS；切换会话只发 `switch` | 避免频繁建连、丢失上下文 |
| **握手期缓冲** | 连接未建立时把消息暂存 `pending`，`onopen` 后按序补发（**有上限**） | 否则页面初始化时发的消息会静默丢失，或数组无界增长 |
| **断线重连 + 补拉** | 指数退避重连；**非首次连接**成功时重新拉会话与消息 | WS 不重放历史，不补拉就会漏消息 |
| **异常收尾** | 断线 / 被踢 / 停止生成时，把「思考中占位气泡」落地、输入区复位 | 否则界面永久卡在「思考中」或输入区永久置灰 |

### 3.7 一个必须理解的细节：`4409` 之后**不能**再自动重连

如果被踢端还自动重连，就会「旧端重连 → 踢掉新端 → 新端重连 → 踢回旧端」，
形成**互踢死循环**。所以前端收到关闭码 `4409` 后：置 `kicked` 标记、**停止自动重连**，
并提示「该账号已在其他窗口登录，本页实时连接已断开；刷新本页可重新接管」。

### 3.8 用户端 / 坐席端（产品视角）

**用户端 `Chat.vue`：多模态输入 → 带图回复 → 参考来源**

| 用户操作 | 前端行为 | 对应后端能力 |
|---|---|---|
| 打字提问 | 乐观渲染自己的消息，立刻插「思考中」占位气泡 | 第 37 节 Agent 编排 |
| 发语音 | `MediaRecorder` 录音 → 上传 → 以 `modality=voice` 发送 | 第 35 节 ASR 转写（服务端） |
| 发截图 | 选图 → 上传 → 随消息发送 | 第 35 节 VLM 诊断 + 第 36 节检索 |
| 等回复 | `bot_stream_delta` 逐字累加到同一条气泡 | 第 37 节流式生成 |
| 看结果 | `bot_stream_end` 定稿：正文 + **图片缩略图** + **参考来源** + 朗读按钮 | 第 36 节 citations / attachments |
| 会话结束 | 输入区置灰，提示「该会话已结束，如需继续请新建会话」 | 只读态 |

**坐席端 `AgentConsole.vue`：一眼看清「该不该接、怎么回」**，右侧是两行两列布局：

| 区域 | 内容 | 来源 |
|---|---|---|
| 待接入队列 | 用户、转接原因、情绪标签 | `handoff_triggered` 事件 |
| AI 摘要 | 问题是什么 / AI 做了什么 / 卡在哪 | 第 37 节 `generate_handoff_summary()` |
| 转接原因 | 触发转人工的规则与依据 | `should_handoff()` 返回的 reason |
| AI 辅助推荐 | 知识卡片 / 相似工单 / 建议话术 | 第 36 节检索 + MCP 查工单 + LLM 草稿 |
| 对话历史 | 用户靠左标「用户」、坐席靠右标「我」 | 同一 `MessageBubble` 按视角切换 |

> 一个工程取舍：摘要与辅助推荐**含 LLM 调用（约 8 秒）**，所以与「对话历史」
> **拆成两个接口**分别加载——否则坐席点开会话要等 8 秒才有任何反馈（详见 4.6）。

### 3.9 源文件职责表（类 / 函数级）

| 层 | 文件 | 主要导出 / 定义 | 职责 |
|---|---|---|---|
| 入口 | `main.js` | — | 创建 app，注册 Element Plus 全部图标，挂载 Pinia / Router |
| 入口 | `App.vue` | — | 根组件，仅 `<router-view>` |
| 路由 | `router/index.js` | `routes`、`router`、`beforeEach` | 路由表 + `meta.roles` 守卫与按角色导流 |
| 状态 | `store/user.js` | `useUserStore`：state(`token`/`user`)、getters(`isLoggedIn`/`role`/`homeRoute`)、actions(`login`/`logout`/`clear`) | 登录态与角色，localStorage 持久化 |
| 网络 | `api/http.js` | `http`(axios)、`mediaUrl()`、请求/响应拦截器、`TOKEN_KEY`/`USER_KEY` | 注入 `Bearer`、401 清理并跳登录、媒体地址补 `token` |
| 网络 | `api/ws.js` | `buildWsUrl()`、`class WSClient`：`on/off/emit/connect/send/flushPending/scheduleReconnect/close` | 按角色建连、事件订阅、握手缓冲、指数退避重连、4409 停止重连 |
| 布局 | `layouts/{User,Agent,Admin}Layout.vue` | `handleLogout()` | 侧边栏 + 顶栏 + `<router-view>` |
| 视图 | `views/Login.vue` | `fillAccount()`、`handleLogin()` | 登录表单与演示账号快填 |
| 视图 | `views/Chat.vue` | `connectWs()`、`handleWsMessage()`、`sendMessage()`、`ensureStreamMsg()`、`finalizeStreaming()`、`uploadFile()`、`startRecording()`/`stopRecording()`、`closeConversation()` | 用户端：多模态输入、流式渲染、会话管理 |
| 视图 | `views/AgentConsole.vue` | `loadQueue()`、`loadAssist()`、`joinSession()`、`openSession()`、`sendReply()`、`returnToAi()`、`endService()`、`startTimer()` | 坐席端：队列 / 摘要 / 辅助推荐（快慢分离）、服务计时 |
| 视图 | `views/admin/Dashboard.vue` | `trendOption` 等 computed、`ws.on('stats_update')` | 数据看板与图表、实时统计 |
| 视图 | `views/admin/Knowledge.vue` | `submitUpload()`、`rebuildIndex()`、`seedKnowledge()`、`runSearch()` | 知识库上传 / 重建 / 检索测试 |
| 视图 | `views/admin/{Tickets,Logs,ToolLogs,Users}.vue` | `loadX()`、`handleSearch()`、详情 / 删除 | 工单 / 对话日志 / 工具日志 / 用户管理 |
| 视图 | `views/admin/Eval.vue` | `startPolling()`、`metricRows()`、`verdictSummary()` | 触发评测、轮询进度、展示指标 |
| 工具 | `utils/format.js` | 时间 / 文本格式化函数 | 展示层格式化 |

### 3.10 可复用组件职责（只看职责）

| 组件 | 职责 |
|---|---|
| `MessageBubble.vue` | 单条气泡：正文 + 图片缩略图 + 参考来源 + 朗读按钮；按视角分侧 |
| `ImageAttachment.vue` | 图片缩略图，点击放大（走鉴权媒体地址） |
| `CitationList.vue` | 参考来源列表（文件 / 页码 / 模态） |
| `ConversationList.vue` | 会话列表，发出 `select` / `new` / `tickets` / `delete` 事件 |
| `ModeBanner.vue` | 模式横幅（`bot` / `waiting_human` / `human`） |
| `BaseChart.vue` | ECharts 封装，供各看板复用 |

---
## Part 4：接口层（REST API）系统讲解
---

> Part 2 讲的是「WS 推」，这里讲「REST 取 / 做」：一个 FastAPI 应用如何被装配起来、
> 五个路由模块如何分工、鉴权与授权如何用依赖注入落地、请求 / 响应 / 错误如何约定，
> 以及媒体访问、接口拆分、后台统计与单源部署。

### 4.1 FastAPI 应用装配：一个 app + 五个 router

`core/main.py` 里只有**一个** `FastAPI(...)` 实例，业务按域拆成五个 `APIRouter`：

| 模块 | `APIRouter(prefix=, tags=)` | 域 |
|---|---|---|
| `auth_api.py` | `/api/auth` · `auth` | 登录 / 登出 / 当前用户 |
| `chat_api.py` | `/api` · `chat` | 会话 / 消息 / 媒体 |
| `agent_api.py` | `/api/agent` · `agent` | 坐席队列 / 会话操作 |
| `admin_api.py` | `/api/admin` · `admin` | 看板 / 知识库 / 评测 / 用户 |
| `ws_api.py` | （无 prefix）· `ws` | 三个 WebSocket 端点 |

在 `main.py` 里逐一 `app.include_router(xxx_api.router)`。`prefix` 决定 URL 前缀，
`tags` 只影响 **OpenAPI 文档分组**——所以服务启动后在 `/docs`（Swagger）能看到按域分组的接口。

> 为什么要拆 router？① 每个文件只管一个域，评审时不用翻大文件；② `tags` 让自动文档自动分组；
> ③ 鉴权等依赖可在 router 级统一声明。

### 4.2 五组路由总览：REST 取 / 做 + WS 推

除 WS 外，业务动作都走 REST。下表由代码**静态提取**（`ast` 读每个模块 `router` 的 `prefix`
与端点），不必手抄文档——接口一变重跑即可对齐（`ws_api` 已在 Part 2 看过）。

In [7]:
# ============================================================
# Cell 7：五组路由一览（ast 静态提取：prefix + 每个端点）
# ============================================================
import ast

MODULES = [
    ("core/api/auth_api.py", "auth"),
    ("core/api/chat_api.py", "chat"),
    ("core/api/agent_api.py", "agent"),
    ("core/api/admin_api.py", "admin"),
    ("core/api/ws_api.py", "ws"),
]

for path, tag in MODULES:
    tree = ast.parse(read(path))
    prefix, routes = "", []
    for node in tree.body:
        if isinstance(node, ast.Assign):
            for tgt in node.targets:
                if getattr(tgt, "id", "") == "router" and isinstance(node.value, ast.Call):
                    for kw in node.value.keywords:
                        if kw.arg == "prefix" and isinstance(kw.value, ast.Constant):
                            prefix = kw.value.value
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            for dec in node.decorator_list:
                if isinstance(dec, ast.Call) and getattr(dec.func, "attr", "") in (
                    "get", "post", "delete", "put", "websocket",
                ):
                    route = dec.args[0].value if dec.args and isinstance(dec.args[0], ast.Constant) else "?"
                    routes.append((dec.func.attr.upper(), prefix + route, node.name))
    print("=" * 70)
    print(f"  {path}  (tags={tag})  共 {len(routes)} 个端点")
    print("=" * 70)
    for method, full, fn in routes:
        print(f"  {method:<9}{full:<34}{fn}()")
    print()

  core/api/auth_api.py  (tags=auth)  共 4 个端点
  POST     /api/auth/login                   login()
  POST     /api/auth/logout                  logout()
  GET      /api/auth/me                      me()
  POST     /api/auth/init                    init()

  core/api/chat_api.py  (tags=chat)  共 8 个端点
  GET      /api/conversations                list_conversations()
  POST     /api/conversations                create_conversation()
  GET      /api/conversations/{conv_id}/messagesget_messages()
  DELETE   /api/conversations/{conv_id}      delete_conversation()
  POST     /api/conversations/{conv_id}/closeclose_conversation()
  GET      /api/tickets                      my_tickets()
  POST     /api/media/upload                 upload_media()
  GET      /api/media/{asset_id}             get_media()

  core/api/agent_api.py  (tags=agent)  共 10 个端点
  GET      /api/agent/queue                  get_queue()
  GET      /api/agent/sessions               get_my_sessions()
  GET      /api/agent/sessi

### 4.3 依赖注入与授权：从密码到角色的一条链

"谁能进哪个接口"分两步：**Authentication（你是谁）** 与 **Authorization（你能不能）**。

| 环节 | 代码 | 做什么 |
|---|---|---|
| 签发凭证 | `core/infra/auth.py::login` | 校验密码 → `secrets.token_urlsafe(32)` 生成 token → 落 `sessions` 表 |
| 还原身份 | `core/infra/auth.py::get_current_user` | 按 token 查 `sessions`，过期则惰性删除，返回用户 |
| 提取凭证 | `core/infra/middleware.py::_extract_token` | 优先 `Authorization: Bearer`，退而支持 `?token=` |
| 登录校验 | `current_user_or_401` | 未登录抛 **401** |
| 角色校验 | `require_role(*roles)` | 角色不在白名单抛 **403** |

> 设计要点：`require_role` 是一个"**依赖工厂**"——调用它本身不校验，而是返回一个
> 闭包 `checker`，真正执行的是 `Depends(require_role("admin"))`。好处是各接口能各自
> 声明白名单，校验逻辑只有一处。

下面打印这条链上的两个关键函数。

In [8]:
# ============================================================
# Cell 8：鉴权链关键源码
# ============================================================
show("core/infra/auth.py", 52, 87, "login —— 校验密码 + 签发 token")
show("core/infra/middleware.py", 65, 77, "require_role —— 依赖工厂（401 → 403）")
print("  记忆点：401 = '你是谁？'，403 = '知道你是谁，但你不能进。'")

  core/infra/auth.py  —— login —— 校验密码 + 签发 token  （第 52~87 行 / 共 121 行）
  52| async def login(username: str, password: str) -> dict:
  53|     """登录：校验账号密码，成功则签发 token 并落库。
  54| 
  55|     参数：
  56|         username: 登录名。
  57|         password: 明文密码。
  58|     返回：
  59|         成功时返回含 token/user_id/username/role/display_name 的 dict；
  60|         失败时返回 {"error": "..."}（用统一结构代替抛异常，方便路由层直接判断）。
  61| 
  62|     边界与异常：
  63|         - 用户不存在 或 密码不匹配：都返回"用户名或密码错误"（故意不区分，防止枚举用户名）；
  64|         - 账号被禁用（status != active）：返回禁用提示，即使密码正确也不放行。
  65|     """
  66|     user = await db.get_user_by_username(username)
  67|     if not user or user["password_hash"] != hash_password(password):
  68|         return {"error": "用户名或密码错误"}
  69|     # 密码对了还要看账号是否被禁用：禁用账号一律拒绝登录。
  70|     if user["status"] != "active":
  71|         return {"error": "账号已被禁用，请联系管理员"}
  72| 
  73|     # secrets.token_urlsafe 生成密码学安全的随机 token（不可预测，适合做凭证）。
  74|     token = secrets.token_urlsafe(32)
  75|     # 过期时间 = 现在 + 配置的

### 4.4 请求 / 响应 / 错误约定

| 方面 | 约定 |
|---|---|
| 请求体 | Pydantic 模型（如登录 `{username, password}`），自动校验，类型错误返回 `422` |
| 响应体 | 统一返回 `dict`（FastAPI 序列化为 JSON），前端拦截器直接取 `data` |
| 业务错误 | 主动抛 `HTTPException(status_code, detail=...)`，前端统一读 `detail` 弹提示 |
| 鉴权错误 | `401`（未登录）/ `403`（角色不符），由 `require_role` 依赖抛出 |
| 文件上传 | `UploadFile = File(...)`，服务端校验大小与 MIME，返回 `asset_id` |
| 文件下载 | `FileResponse`，先过鉴权再回文件（见 4.5） |

> 一个细节：`http.js` 响应拦截器 `return response.data`，所以业务层拿到的就是后端 dict，
> 不用再写 `res.data.data`。

### 4.5 媒体：上传一次，鉴权访问

多模态消息里的图片 / 语音**不随 WS 传输原始字节**，而是先走 REST 上传拿到 `asset_id`，
再随消息发送。读取时也不是公开静态目录，而是**带鉴权的接口**：

| 步骤 | 端点 | 鉴权 | 说明 |
|---|---|---|---|
| 上传 | `POST /api/media/upload` | user / agent / admin | 限制大小，按 MIME 推断 `image` / `voice` |
| 访问 | `GET /api/media/{asset_id}` | user / agent / admin | 返回 `FileResponse`；不存在则 404 |

> 媒体为什么不放静态目录？因为截图可能含内部信息，不能让拿到 URL 的匿名者直接下载。
> 前端 `<img>` / `<audio>` 无法自定义请求头，所以媒体地址会带 `?token=`——
> 与 WS 握手的处理同源（见 5.5 练习二）。

In [9]:
# ============================================================
# Cell 9：媒体上传 / 鉴权访问源码
# ============================================================
import ast

tree = ast.parse(read("core/api/chat_api.py"))
print("=" * 70)
print("  core/api/chat_api.py —— 媒体相关端点")
print("=" * 70)
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        for dec in node.decorator_list:
            if isinstance(dec, ast.Call) and getattr(dec.func, "attr", "") in ("get", "post"):
                route = dec.args[0].value if dec.args else "?"
                if "media" in route:
                    print(f"  {dec.func.attr.upper():<6}{route:<22}{node.name}()")
print()

show("core/api/chat_api.py", 180, 210, "upload_media —— 上传 + 大小/类型校验")
show("core/api/chat_api.py", 213, 226, "get_media —— 鉴权访问 + FileResponse")

  core/api/chat_api.py —— 媒体相关端点
  POST  /media/upload         upload_media()
  GET   /media/{asset_id}     get_media()

  core/api/chat_api.py  —— upload_media —— 上传 + 大小/类型校验  （第 180~210 行 / 共 226 行）
 180| @router.post("/media/upload")
 181| async def upload_media(
 182|     file: UploadFile = File(...),
 183|     user: dict = Depends(require_role("user", "agent", "admin")),
 184| ) -> dict:
 185|     """上传媒体文件（图片或语音）。
 186| 
 187|     调用方：各端发送带图/语音消息前先上传。
 188|     入参：multipart 表单字段 file（UploadFile），File(...) 表示该字段必填。
 189|     鉴权：允许 user / agent / admin。
 190|     错误码：400 —— 文件为空或超过 settings.KB_MAX_FILE_MB 大小限制。
 191|     返回：{"asset_id", "url", "modality"}；modality 按 MIME 类型推断为 image / voice。
 192|     """
 193|     # 依赖注入已在进入函数体前完成登录与角色校验，因此此处读文件时调用者必定合法
 194|     data = await file.read()
 195|     if not data:
 196|         raise HTTPException(status_code=400, detail="上传文件为空")
 197|     if len(data) > settings.KB_MAX_FILE_MB * 1024 * 1024:
 198|         raise HTTPException(status

### 4.6 一个真实取舍：坐席接口为什么拆成两个

坐席点开一个会话，需要"对话历史"和"AI 摘要 + 辅助推荐"两类数据。后者两项各含一次
LLM 调用，合计约 **8 秒**。若塞进同一个接口，坐席点了会话要等 8 秒才有任何反应——
实际体验就是"点不动"。所以拆成两个端点，让"快"的先到、"慢"的后补：

| 端点 | 内容 | 耗时 | 前端策略 |
|---|---|---|---|
| `GET /session/{id}` | 会话 + 对话历史 | 毫秒级 | 点击后立即请求并渲染 |
| `GET /session/{id}/assist` | 摘要 + 辅助推荐 | 约 8 秒 | 并行请求，到了再填充右侧 |

> 两项互相独立，用 `asyncio.gather` 并行，把延迟从"相加"变成"取较大者"；
> 并按消息数缓存，坐席来回切换同一会话时秒回、不再重复调用模型。

In [10]:
# ============================================================
# Cell 10：坐席会话接口拆分（快 / 慢分离）
# ============================================================
import ast

tree = ast.parse(read("core/api/agent_api.py"))
print("=" * 70)
print("  core/api/agent_api.py —— 会话相关端点")
print("=" * 70)
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        for dec in node.decorator_list:
            if isinstance(dec, ast.Call) and getattr(dec.func, "attr", "") in ("get", "post"):
                route = dec.args[0].value if dec.args else "?"
                if "/session/" in route:
                    print(f"  {dec.func.attr.upper():<6}{route:<30}{node.name}()")
print()

show("core/api/agent_api.py", 95, 108, "get_session —— 只回会话与历史（快）")
show("core/api/agent_api.py", 111, 138, "get_session_assist —— 摘要+推荐（慢，按消息数缓存）")

  core/api/agent_api.py —— 会话相关端点
  GET   /session/{conv_id}            get_session()
  GET   /session/{conv_id}/assist     get_session_assist()
  POST  /session/{conv_id}/join       join()
  POST  /session/{conv_id}/reply      reply()
  POST  /session/{conv_id}/return_to_aireturn_to_ai()
  POST  /session/{conv_id}/end_serviceend_service()

  core/api/agent_api.py  —— get_session —— 只回会话与历史（快）  （第 95~108 行 / 共 251 行）
  95| 
  96| @router.get("/session/{conv_id}")
  97| async def get_session(conv_id: str, user: dict = Depends(require_role("agent", "admin"))) -> dict:
  98|     """会话基本信息（对话历史）。
  99| 
 100|     刻意不含 AI 摘要与辅助建议：那两项各含一次 LLM 调用，合计约 8 秒；
 101|     若塞在本接口里，坐席点击列表后要等 8 秒才有任何反馈，形同「点不动」。
 102|     它们由 GET /session/{id}/assist 异步补齐（见 12.4）。
 103|     """
 104|     conversation = await db.get_conversation(conv_id)
 105|     if not conversation:
 106|         raise HTTPException(status_code=404, detail="会话不存在")
 107|     _ensure_viewable(conversation, user)
 108|     messages = awa

### 4.7 管理后台与统计口径

管理端是"接口大户"（看板、日志、工单、用户、工具、知识库、评测）。统计指标在
`core/service/analytics.py` 里被**冻结成明确口径**，避免同一个词在不同人那里算法不同：

| 指标 | 口径 |
|---|---|
| 会话总数 | 周期内新建会话数（所有比率的分母） |
| 解决率 | `end_reason='resolved'` / 会话总数 |
| 转人工率 | 发生过 handoff 的会话数 / 会话总数 |
| 平均响应时间 | Σ(用户消息 → AI 首次回复) / 消息对数（仅 bot 模式） |
| 平均人工处理时长 | Σ(坐席接入 → 会话结束) / 人工会话数 |

> 看板不是前端轮询，而是 `push_stats_to_admins()` 通过 `/ws/admin` **主动推送**；
> 推送用的 `snapshot_for_ws()` 只取 4 个计数器，因为高频推送必须足够轻。

In [11]:
# ============================================================
# Cell 11：管理端点清单 + 统计口径（从源码自证）
# ============================================================
import ast

tree = ast.parse(read("core/api/admin_api.py"))
count, groups = 0, {}
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        for dec in node.decorator_list:
            if isinstance(dec, ast.Call) and getattr(dec.func, "attr", "") in ("get", "post", "delete", "put"):
                count += 1
                route = dec.args[0].value if dec.args else "?"
                parts = [p for p in route.split("/") if p]
                groups.setdefault(parts[0] if parts else route, []).append(route)
print("=" * 70)
print(f"  core/api/admin_api.py —— 共 {count} 个端点，按一级路径分组")
print("=" * 70)
for key in sorted(groups):
    print(f"  /{key:<16}{len(groups[key])} 个端点")
print()

# 指标口径直接从源码文档读取，避免"文档与代码脱节"
analytics_src = read("core/service/analytics.py")
doc = ast.get_docstring(ast.parse(analytics_src)) or ""
print("=" * 70)
print("  analytics.py 冻结的指标口径")
print("=" * 70)
for line in doc.splitlines():
    if line.strip().startswith("- "):
        print("   ", line.strip())
print()

# 实时推送只取 4 个计数器：从 snapshot_for_ws 的源码里把字段名取出来
KEY_PAT = r'"([a-z_]+)": '
seg = analytics_src[analytics_src.index("async def snapshot_for_ws"):]
seg = seg[:seg.index("async def push_stats_to_admins")]
print(f"  /ws/admin 高频推送字段：{re.findall(KEY_PAT, seg)}")

  core/api/admin_api.py —— 共 29 个端点，按一级路径分组
  /dashboard       1 个端点
  /eval            6 个端点
  /knowledge       8 个端点
  /logs            3 个端点
  /tickets         4 个端点
  /tools           2 个端点
  /users           5 个端点

  analytics.py 冻结的指标口径
    - 会话总数：统计周期内新建会话数（所有比率的分母）；
    - 解决率：(end_reason='resolved') / 会话总数；
    - 转人工率：发生过 handoff 的会话数 / 会话总数；
    - 平均响应时间：Σ(用户消息 → AI 首次回复) / 消息对数（仅 bot 模式）；
    - 平均人工处理时长：Σ(坐席接入 → 会话结束) / 人工会话数。

  /ws/admin 高频推送字段：['session_total', 'today_sessions', 'active_sessions', 'waiting_human']


### 4.8 单源部署：为什么 :8000 也能打开前端

后端 `main.py` 同时伺服前端构建产物：`/assets/*` 挂静态目录，其余路径走 `spa_fallback`——
**先判断文件是否存在，再判断保留前缀（`api/`、`ws/`），最后回落到 `index.html`**。

顺序**很关键**：若先判前缀，新加的前端静态文件可能被误判成 API 而 404。
另外 `.html` 一律带 `Cache-Control: no-cache`——否则浏览器会复用**旧 HTML 外壳**
去加载**已删除的旧 chunk**，页面就会莫名失灵。

In [12]:
# ============================================================
# Cell 12：SPA fallback 源码（单源部署的关键）
# ============================================================
show("core/main.py", 139, 167, "前端静态资源挂载 + SPA fallback")

  core/main.py  —— 前端静态资源挂载 + SPA fallback  （第 139~167 行 / 共 172 行）
 139| # ---- 前端静态资源（如已构建；开发期由 Vite 3000 端口提供）----
 140| _dist = PROJECT_ROOT / "frontend" / "dist"
 141| if _dist.exists():
 142|     if (_dist / "assets").exists():
 143|         app.mount("/assets", StaticFiles(directory=_dist / "assets"), name="assets")
 144| 
 145|     @app.get("/{full_path:path}")
 146|     async def spa_fallback(full_path: str):
 147|         # 顺序很关键：先判断文件是否存在，再判断保留前缀，最后回落到 index.html
 148|         if full_path:
 149|             candidate = (_dist / full_path).resolve()
 150|             if candidate.is_file() and _dist.resolve() in candidate.parents:
 151|                 # HTML 外壳一律禁用缓存，避免浏览器复用旧外壳加载已删除的 chunk
 152|                 if candidate.suffix == ".html":
 153|                     return FileResponse(candidate, headers={"Cache-Control": "no-cache"})
 154|                 return FileResponse(candidate)
 155|         if full_path.startswith(("api/", "ws/")):
 156|             return JSONR

---
## Part 5：联调与稳定性
---

### 5.1 联调第一招：把"事件契约"对一遍

前后端只靠 `type` 字符串约定通信。**最有效的联调手段**是：
把后端发出的事件集合 与 前端处理的事件集合**都提取出来做差集**。

In [13]:
# ============================================================
# Cell 13：前后端事件契约对齐（差集检查）
# ============================================================
backend_types = set()
for p in BACKEND_FILES:
    backend_types |= set(type_pat.findall(read(p)))

client_types = set()
for p in Path("frontend/src").rglob("*"):
    if p.suffix in (".js", ".vue"):
        t = read(str(p))
        client_types |= set(re.findall(r"case '([a-z_]+)'", t))
        client_types |= set(re.findall(r"\.on\('([a-z_]+)'", t))
client_types -= {"message", "open", "close", "error", "kicked"}   # WSClient 自身的连接事件

print("=" * 68)
print("  事件契约对齐")
print("=" * 68)
print(f"  后端发出：{len(backend_types)} 个")
print(f"  前端处理：{len(client_types)} 个")
print()
orphan_client = sorted(client_types - backend_types)
orphan_backend = sorted(backend_types - client_types)
print(f"  [风险] 前端处理但后端从不发出：{orphan_client or '无'}")
print(f"  [提示] 后端发出但前端未处理：{orphan_backend or '无'}")
print()
print("  对差集的解读：")
print("    - '前端处理但后端未发' 若不为空，说明有死代码或拼写错误，必须修；")
print("    - '后端发出但前端未处理' 不一定是 Bug：")
print("        pong         -> 心跳应答，前端无需处理；")
print("        agent_status -> 广播给其他坐席，工作台通过自身状态与 queue_update 展示。")
print("        （stats_update 由管理端 Dashboard.vue 的 ws.on(...) 处理，故不在差集里）")


  事件契约对齐
  后端发出：16 个
  前端处理：14 个

  [风险] 前端处理但后端从不发出：无
  [提示] 后端发出但前端未处理：['agent_status', 'pong']

  对差集的解读：
    - '前端处理但后端未发' 若不为空，说明有死代码或拼写错误，必须修；
    - '后端发出但前端未处理' 不一定是 Bug：
        pong         -> 心跳应答，前端无需处理；
        agent_status -> 广播给其他坐席，工作台通过自身状态与 queue_update 展示。
        （stats_update 由管理端 Dashboard.vue 的 ws.on(...) 处理，故不在差集里）


### 5.2 常见坑与答疑（都是真实踩过的）

| 现象 | 根因 | 修法 |
|------|------|------|
| 老用户收不到新消息，但连接是好的 | 旧连接 `finally` **无条件 pop**，误删了重连的新连接 | 注销校验身份（`is ws`） |
| 同账号两处登录互相踢 | 被踢端自动重连，把新端顶掉 | `4409` 关闭码 + 前端置 `kicked` 停止重连 |
| 点「停止」后气泡永远"思考中" | 取消后没有收尾占位气泡 | 统一 `finalizeStreaming()` 收尾 |
| 坐席点会话"点不动" | 摘要（LLM）与基础数据塞在同一接口 | 拆成 `session/{id}` 与 `session/{id}/assist` |
| 断线重连后消息丢失 | 重连不重放历史 | `on('open')` 非首次连接时补拉消息与状态 |
| 重连后 token 白烧 | 每次重连都重跑摘要/推荐 | 重连时 `withAssist=false` |
| 页面能开但输入框打不了字 | 浏览器缓存了**旧 HTML 外壳**，去加载已删除的旧 chunk | `.html` 响应加 `Cache-Control: no-cache` |

### 5.3 实战任务

| 任务 | 做法 | 观察点 |
|------|------|--------|
| **走通 AI 带图回复** | 用户发截图提问 | 收到 `bot_stream_delta`→`bot_stream_end`，附件渲染缩略图、显示参考来源 |
| **走通转人工闭环** | 用户说"转人工"，坐席接入再"转回 AI" | 用户端 mode 横幅 `waiting_human`→`human`→`bot` |
| **模拟断线补拉** | 断开网络数秒再恢复 | 重连后补齐消息与状态，不重复跑摘要 |
| **双开互踢** | 同账号在两窗口登录 | 先连者收到 `4409` 提示且不再重连 |

### 5.4 随堂练习一：转人工时，用户端与坐席端各收到哪些事件？

请按顺序列出"用户说转人工 → 坐席接入 → 坐席转回 AI"这条链路上，
**用户端**与**坐席端**分别收到的事件，并说明每条事件由哪个后端模块发出。

In [14]:
# ============================================================
# Cell 14：练习一参考思路（含答案）
# ============================================================
answer = """
【用户说"转人工"】
  · 关键词命中 -> session_flow.gate_intent 零 LLM 短路 -> session_router.trigger_handoff()
  · 用户端收到 handoff_triggered（"正在为您转接人工客服，请稍候…"，mode=waiting_human）
  · 若有坐席：坐席端收到 handoff_triggered（带 user_name / handoff_reason / summary 交接摘要）
  · 若无坐席：用户端随后收到 return_to_bot（含工单号 ticket_id），mode 回到 bot

【坐席接入】
  · 用户端收到 agent_joined（"人工客服已接入"，mode=human）
  · 坐席端收到 queue_update（队列变化，把该会话从"待接入"移入"服务中"）

【坐席转回 AI】
  · 用户端收到 return_to_bot（"本次人工服务已结束，已转回智能客服"，可带 note）
  · 坐席端收到 queue_update（returned）

事件发出方：以上 user/agent 侧的推送都由 core/service/session_router.py 经
core/infra/ws_manager.py 广播；ws_api.py 负责端点与消息循环。
—— 记忆要点：**用户端关心"我现在是谁在服务"，坐席端关心"队列变了没"。**
"""
print(answer)


【用户说"转人工"】
  · 关键词命中 -> session_flow.gate_intent 零 LLM 短路 -> session_router.trigger_handoff()
  · 用户端收到 handoff_triggered（"正在为您转接人工客服，请稍候…"，mode=waiting_human）
  · 若有坐席：坐席端收到 handoff_triggered（带 user_name / handoff_reason / summary 交接摘要）
  · 若无坐席：用户端随后收到 return_to_bot（含工单号 ticket_id），mode 回到 bot

【坐席接入】
  · 用户端收到 agent_joined（"人工客服已接入"，mode=human）
  · 坐席端收到 queue_update（队列变化，把该会话从"待接入"移入"服务中"）

【坐席转回 AI】
  · 用户端收到 return_to_bot（"本次人工服务已结束，已转回智能客服"，可带 note）
  · 坐席端收到 queue_update（returned）

事件发出方：以上 user/agent 侧的推送都由 core/service/session_router.py 经
core/infra/ws_manager.py 广播；ws_api.py 负责端点与消息循环。
—— 记忆要点：**用户端关心"我现在是谁在服务"，坐席端关心"队列变了没"。**



### 5.5 随堂练习二：为什么 token 要放在 WebSocket 的 URL 里？

浏览器建立 WebSocket 时能不能像 axios 那样自定义 `Authorization` 头？
如果不能，后端是怎么拿到 token 的？这样做有什么代价？

In [15]:
# ============================================================
# Cell 15：练习二参考思路（含答案）
# ============================================================
answer = """
1) 不能。浏览器 WebSocket 构造函数只接受 URL 与可选的子协议（protocols），
   没有 headers 参数；因此无法像 axios / fetch 那样设置 Authorization 头。

2) 本项目的做法：
   - 前端把 token 拼在 URL 查询参数里：/ws/user?token=xxx（api/ws.js）
   - 后端在握手时从查询参数取：ws.query_params.get("token")（ws_api._authenticate）
   - 未登录 close(4401)，角色不符 close(4403)。

3) 代价与缓解：
   - URL 可能被网关/代理日志记录，token 有泄漏面 —— 缓解：HTTPS/WSS 加密传输、
     token 有过期时间（TOKEN_EXPIRE_HOURS）、退出登录即作废（服务端 sessions 表）；
   - 生产环境建议用短时效 token，或用 Sec-WebSocket-Protocol 传票。

替代方案：用子协议传 token（不进 URL），但浏览器/代理兼容性更麻烦。
"""
print(answer)


1) 不能。浏览器 WebSocket 构造函数只接受 URL 与可选的子协议（protocols），
   没有 headers 参数；因此无法像 axios / fetch 那样设置 Authorization 头。

2) 本项目的做法：
   - 前端把 token 拼在 URL 查询参数里：/ws/user?token=xxx（api/ws.js）
   - 后端在握手时从查询参数取：ws.query_params.get("token")（ws_api._authenticate）
   - 未登录 close(4401)，角色不符 close(4403)。

3) 代价与缓解：
   - URL 可能被网关/代理日志记录，token 有泄漏面 —— 缓解：HTTPS/WSS 加密传输、
     token 有过期时间（TOKEN_EXPIRE_HOURS）、退出登录即作废（服务端 sessions 表）；
   - 生产环境建议用短时效 token，或用 Sec-WebSocket-Protocol 传票。

替代方案：用子协议传 token（不进 URL），但浏览器/代理兼容性更麻烦。



### 5.6 随堂练习三：为什么 `4409` 之后前端必须停止自动重连？

请说明：若被踢端仍然自动重连，会发生什么？正确的前端行为是什么？

In [16]:
# ============================================================
# Cell 16：练习三参考思路（含答案）
# ============================================================
answer = """
若被踢端仍自动重连，会形成【互踢死循环】：
  旧端被踢 -> 自动重连 -> 服务端把"新端"踢掉 -> 新端又自动重连 -> 反踢旧端 -> ...
两台设备（或两个窗口）都没法稳定使用。

正确行为（本项目已实现）：
  1. 服务端：register_agent/register_user 发现同账号已有连接时，
     用自定义关闭码 4409（WS_CLOSE_KICKED）+ 原因"账号已在别处登录"关闭旧连接；
  2. 前端：ws.js 的 onclose 里判断 ev.code === 4409 -> 置 kicked=true、清空 pending、
     emit('kicked') 并 return（不再 scheduleReconnect）；
  3. UI：弹出提示"该账号已在其他窗口登录，本页实时连接已断开；刷新本页可重新接管"，
     并把未完成的流式气泡收尾、输入区复位。

一句话：**新连接赢、旧连接体面退出**——这是"同账号多端登录"唯一稳定的策略。
"""
print(answer)


若被踢端仍自动重连，会形成【互踢死循环】：
  旧端被踢 -> 自动重连 -> 服务端把"新端"踢掉 -> 新端又自动重连 -> 反踢旧端 -> ...
两台设备（或两个窗口）都没法稳定使用。

正确行为（本项目已实现）：
  1. 服务端：register_agent/register_user 发现同账号已有连接时，
     用自定义关闭码 4409（WS_CLOSE_KICKED）+ 原因"账号已在别处登录"关闭旧连接；
  2. 前端：ws.js 的 onclose 里判断 ev.code === 4409 -> 置 kicked=true、清空 pending、
     emit('kicked') 并 return（不再 scheduleReconnect）；
  3. UI：弹出提示"该账号已在其他窗口登录，本页实时连接已断开；刷新本页可重新接管"，
     并把未完成的流式气泡收尾、输入区复位。

一句话：**新连接赢、旧连接体面退出**——这是"同账号多端登录"唯一稳定的策略。



### 5.7 本节小结

| 环节 | 文件 | 核心设计 |
|------|------|----------|
| 前端轮廓 | `router/index.js` | 一个路由表 + `meta.roles` 分权；菜单与守卫同源 |
| 事件契约 | 后端多处 | 前端只认 `type`；`bot_stream_delta/end` 承载 Agent 流式产出 |
| WS 服务端 | `core/api/ws_api.py` | 三端点分角色；握手鉴权（query token）；后台任务 + 停止生成 |
| 连接管理 | `core/infra/ws_manager.py` | 分组广播；互踢带 `4409`；注销**校验身份**（防误删新连接） |
| 前端接收 | `api/ws.js`、`views/*` | 一条连接复用；pending 缓冲；重连补拉；异常收尾；`4409` 停止重连 |
| 产品呈现 | `views/Chat.vue`、`AgentConsole.vue` | 用户端带图回复+来源；坐席端两行两列 + 摘要/辅助推荐 |
| 接口层 | `core/api/*.py` | 四组 REST + 一组 WS；`require_role` 统一鉴权 |
| 部署 | `core/main.py` | 单源部署 + SPA fallback（先判文件、再判前缀）+ HTML 不缓存 |
| 联调 | — | 前后端**事件契约差集**检查；异常场景清单 |

### 5.8 交付物 / 验收

- 三端页面可用、权限正确（守卫与菜单同源）；
- 「用户发图 → AI 带图回复」与「转人工 → 坐席接入 → 转回 AI」两条实时链路跑通；
- 网络闪断后重连补拉、不重复跑摘要；同账号双开先连者被踢并提示且不再重连；
- 前端处理的事件集合与后端发出集合对齐（无"前端处理但后端不发"的孤儿事件）。

### 5.9 下节课预告

**第 39 节：多模态全栈评测、企业级部署与商业化交付**
- 评测体系：ASR/TTS/VLM/检索四套件 + 指标 + LLM 评判 + 结果落库；
- 回归测试：`pytest` 全绿是交付门槛；
- 容器化部署：两个镜像 + Compose 编排 + 数据卷持久化；
- 商业化交付：产品说明书、技术文档、部署运维手册、评测报告。

---

## 📚 参考资料

- 代码：`frontend/src/router/index.js` —— 路由表与 `meta.roles` 权限
- 代码：`frontend/src/api/ws.js` —— WebSocket 封装（重连 / 缓冲 / 4409）
- 代码：`frontend/src/views/Chat.vue` —— 用户端（多模态输入 / 流式渲染）
- 代码：`frontend/src/views/AgentConsole.vue` —— 坐席端（队列 / 摘要 / 辅助推荐）
- 代码：`frontend/src/components/*` —— 可复用组件
- 代码：`core/api/ws_api.py` —— 三个 WebSocket 端点
- 代码：`core/infra/ws_manager.py` —— 连接管理与广播
- 代码：`core/api/{auth,chat,agent,admin}_api.py` —— 四组 REST 路由
- 代码：`core/main.py` —— 单源部署与 SPA fallback
- 代码：`core/infra/auth.py` —— 服务端会话式鉴权（token 签发 / 校验）
- 代码：`core/infra/middleware.py` —— `require_role` 依赖工厂（区分 401 / 403）
- 代码：`core/service/analytics.py` —— 看板统计口径与 `/ws/admin` 推送

> 🎉 恭喜你完成了本节课的学习！
> 现在你不仅知道三端长什么样，更知道
> **Agent 的产出是怎么变成事件送达用户与坐席的、为什么连接注销必须校验身份、
> 为什么 4409 之后不能再重连**。
- 代码：`core/api/*.py` —— 五组 REST 路由与依赖注入装配（见 Part 4）
